# PUMA V13.2 — 03 Evaluate / Final Train / Infer

Recommended flow: lock the reviewed **100-epoch development winner**, retrain it on all
labeled ROIs for 100 epochs, then run local or Grand-Challenge inference. Every step past
the ranking table is behind an explicit switch, so Run All is safe.


In [ ]:
# Project bootstrap. Runs on the "SymbioPan (uv .venv)" kernel on this workstation, and
# on Colab without changes. No %pip here: dependencies come from setup_local.sh (uv) or,
# on Colab, from `!pip install -q -r requirements_colab.txt` in a scratch cell.
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Research/PUMA')
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = Path.cwd().resolve()
    ON_COLAB = False

PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
if not (PROJECT_DIR / 'puma').is_dir():
    raise RuntimeError(
        f"{PROJECT_DIR} is not the project root (no puma/ package here). "
        "Start JupyterLab from the project root, or set PROJECT_DIR explicitly."
    )
os.chdir(PROJECT_DIR)

# Drop any stale puma modules so an edited package is always re-imported.
for module_name in [m for m in sys.modules if m == 'puma' or m.startswith('puma.')]:
    del sys.modules[module_name]
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PUMA_STAGE2_CROP_CACHE_MB', '512')
os.environ.setdefault('PUMA_V132_AUTO_OOM_FALLBACK', '1')

# GPU selection. On a workstation with two or more GPUs this pins training to GPU 1,
# leaving GPU 0 for the display and for other jobs; with a single GPU (or on Colab) it
# falls back to GPU 0. It must happen before torch is imported, because
# CUDA_VISIBLE_DEVICES is only read when the CUDA driver initialises -- puma.gpu imports
# no torch for that reason. An existing CUDA_VISIBLE_DEVICES is respected, so
#     CUDA_VISIBLE_DEVICES=0 ./.venv/bin/jupyter lab
# still overrides this. The selected GPU becomes cuda:0 inside torch.
from puma.gpu import describe_selection, select_cuda_device

PREFERRED_GPU_INDEX = 1
gpu_selection = select_cuda_device(PREFERRED_GPU_INDEX)

print('PROJECT_DIR =', PROJECT_DIR)
print('python      =', sys.executable)
print()
print(describe_selection(gpu_selection))


In [ ]:
from puma.runtime import create_runtime, resolve_hf_token

DEVELOPMENT_EPOCH_PROFILE = 50  # screening comparison; the winner is retrained separately at 100

runtime = create_runtime(
    PROJECT_DIR,
    run_folds=(0, 1, 2, 3, 4),
    seeds=(0,),
    stage1_epochs=40,
    stage2_epochs=DEVELOPMENT_EPOCH_PROFILE,
    stage1_effective_batch_size=16,
    stage2_effective_batch_size=256,
    stage1_micro_batch_size=16,
    stage2_micro_batch_size=256,
    preprocessing_workers=0,  # 0 = all logical CPU cores
    early_stopping_enabled=False,
)
runtime.training.number_of_workers = 4
runtime.training.deterministic = False
HF_TOKEN = resolve_hf_token()
print(runtime.as_dict())


In [ ]:
from puma.pipeline.experiments_v132 import aggregate_v132_results
from puma.stage2.catalog import VERSION132_EXPERIMENTS

ranking = aggregate_v132_results(
    runtime, VERSION132_EXPERIMENTS, epoch_profile=DEVELOPMENT_EPOCH_PROFILE
)
if ranking.empty:
    print('No complete V13.2 results for this epoch profile.')
else:
    sort_cols = [c for c in ('macro_f1', 'conditional_type_macro_f1_present', 'reject_f1')
                 if c in ranking.columns]
    display(ranking.sort_values(sort_cols, ascending=False, na_position='last'))


In [ ]:
CREATE_DEVELOPMENT_LOCK = False
SELECTED_EXPERIMENT = 'V13_2_02_META_RARE_BS'  # replace if another experiment wins

if CREATE_DEVELOPMENT_LOCK:
    from puma.pipeline.experiments_v132 import lock_v132_winner

    stage2_lock = lock_v132_winner(
        runtime,
        selected_experiment=SELECTED_EXPERIMENT,
        candidate_experiments=(SELECTED_EXPERIMENT,),
    )
    print(stage2_lock)


In [ ]:
TRAIN_FINAL_MODEL = False
FORCE_FINAL_RETRAIN = False
FINAL_EPOCHS = 100  # allowed: 50 or 100; 100 is recommended for submission

if TRAIN_FINAL_MODEL:
    from puma.pipeline.final_v132 import train_final_stage2_v132

    final_lock = train_final_stage2_v132(
        runtime,
        hf_token=HF_TOKEN,
        selected_experiment=SELECTED_EXPERIMENT,
        final_epochs=FINAL_EPOCHS,
        force=FORCE_FINAL_RETRAIN,
        auto_oom_fallback=True,
    )
    print(final_lock['final_checkpoint'])


In [ ]:
CHECK_FINAL_READY = False
if CHECK_FINAL_READY:
    from puma.pipeline.final_v132 import final_v132_ready

    print(final_v132_ready(runtime))


In [ ]:
RUN_LOCAL_INFERENCE = False
INFERENCE_DIR = PROJECT_DIR / 'Dataset' / 'challenge_test_images'

if RUN_LOCAL_INFERENCE:
    from puma.pipeline.inference import run_inference

    print(run_inference(runtime, input_dir=INFERENCE_DIR, hf_token=HF_TOKEN))


In [ ]:
# Grand-Challenge contract test/run.
RUN_GRAND_CHALLENGE_INFERENCE = False
GC_INPUT_IMAGE = None  # None uses /input/images/melanoma-wsi/<uuid>.tif
GC_TISSUE_MASK = None  # optional externally generated 0..5 tissue mask

if RUN_GRAND_CHALLENGE_INFERENCE:
    from puma.pipeline.inference import run_grand_challenge_inference

    summary = run_grand_challenge_inference(
        runtime,
        input_image=GC_INPUT_IMAGE,
        output_root=Path('/output'),
        tissue_mask_source=GC_TISSUE_MASK,
        allow_background_tissue_fallback=True,
        hf_token=HF_TOKEN,
    )
    print(summary)
